<a href="https://colab.research.google.com/github/Ameb8/sudoku-ranker/blob/main/SudokuRanker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Sudoku Ranker

This notebook ranks the difficulty of sudoku puzzle based of the human strategies required to solve them. Once the needed strategies have been determined, a feed-forward neural network is used to assign a numerical difficulty rank to the puzzle.

In [2]:
import numpy as np
import pandas as pd
import requests

In [3]:
def init_candidates(board):
  candidates = np.ones((9, 9, 9), dtype=bool)

  for row in range(9):
    for col in range(9):
      if board[row, col] != 0:
        update_candidates(candidates, row, col, board[row, col])

  return candidates

### Update Candidates After Value Placement

In [4]:
def update_candidates(candidates, row, col, val):
  """
  Updates possible candidates after placing val at (row, col).

  candidates: 9x9x9 ndarray of booleans
  row, col: int (0-8)
  val: int (1-9)
  """
  digit_idx = val - 1

  # Set only placed digit as possible in this cell
  candidates[row, col, :] = False
  candidates[row, col, digit_idx] = True

  # Eliminate val as candidate from same row and column
  candidates[row, :, digit_idx] = False
  candidates[:, col, digit_idx] = False

  # Eliminate val as candidate from subsquare
  box_row_start = (row // 3) * 3
  box_col_start = (col // 3) * 3
  candidates[box_row_start:box_row_start+3, box_col_start:box_col_start+3, digit_idx] = False

  # Restore True for placed value in its own cell
  candidates[row, col, digit_idx] = True


### Class for Determining All Locations in Subsections Containing Specific Location

In [5]:
class CellGroup:
  def __init__(self, row, col):
    self.row = [(row, c) for c in range(9)]
    self.col = [(r, col) for r in range(9)]
    box_row = (row // 3) * 3
    box_col = (col // 3) * 3
    self.sqr = [(box_row + r, box_col + c) for r in range(3) for c in range(3)]

# Human Strategies for Solving Sudoku

### Naked Singles

In [6]:
def solve_naked_singles(board, candidates):
  """
  Solves the naked singles strategy as far as possible.

  board: 9x9 ndarray of ints
  candidates: 9x9x9 ndarray of booleans
  returns: number of cells filled
  """
  filled_cells = 0
  while True:
    num_filled = naked_singles(board, candidates)
    if num_filled == 0:
      break
    filled_cells += num_filled

  return filled_cells

In [7]:
def naked_singles(board, candidates):
  """
  Solves the naked singles strategy.

  board: 9x9 ndarray of ints
  candidates: 9x9x9 ndarray of booleans
  returns: number of cells filled
  """
  can_fill = []

  for i in range(9): # Iterate rows
    for j in range(9): # Iterate columns
      if board[i, j] != 0:
        continue # Cell filled, continue
      num_candidates = 0
      for k in range(9): # Iterate candidates
        if candidates[i, j, k]:
          num_candidates += 1
          digit_idx = k
      if num_candidates == 1: # Only 1 possibility, save
        can_fill.append((i, j, digit_idx))

  # Fill valid cells
  for i, j, k in can_fill:
    board[i, j] = k + 1
    update_candidates(candidates, i, j, k + 1)

  return len(can_fill)


### Hidden Singles

In [8]:
def solve_hidden_singles(board, candidates):
  """
  Solves the hidden singles strategy.

  board: 9x9 ndarray of ints
  candidates: 9x9x9 ndarray of booleans
  """
  filled = 0

  # Check rows
  for digit in range(9):
    for row in range(9):
      position = [(row, col) for col in range(9) if board[row, col] == 0 and candidates[row, col, digit]]
      if len(position) == 1:
        r, c = position[0]
        board[r, c] = digit + 1
        update_candidates(candidates, r, c, digit + 1)
        filled += 1

  # Check columns
  for digit in range(9):
    for col in range(9):
      position = [(row, col) for row in range(9) if board[row, col] == 0 and candidates[row, col, digit]]
      if len(position) == 1:
        r, c = position[0]
        board[r, c] = digit + 1
        update_candidates(candidates, r, c, digit + 1)
        filled += 1

  # Check boxes
  for digit in range(9):
    for box_row in range(3):
      for box_col in range(3):
        positions = []
        for i in range(3):
          for j in range(3):
            r = box_row * 3 + i
            c = box_col * 3 + j
            if board[r, c] == 0 and candidates[r, c, digit]:
              positions.append((r, c))
        if len(positions) == 1:
          r, c = positions[0]
          board[r, c] = digit + 1
          update_candidates(candidates, r, c, digit + 1)
          filled += 1

  return filled


### Naked Pairs

In [9]:
def solve_naked_pairs(board, candidates):
  """
  Solves puzzle with the naked pairs strategy.

  board: 9x9 ndarray of ints
  candidates: 9x9x9 ndarray of booleans
  """
  total_elims = 0

  for unit_index in range(9):
    row_group = [(unit_index, col) for col in range(9)]
    col_group = [(row, unit_index) for row in range(9)]
    sqr_group = CellGroup(unit_index // 3 * 3, unit_index % 3 * 3).sqr


    total_elims += solve_naked_pairs_group(candidates, row_group)
    total_elims += solve_naked_pairs_group(candidates, col_group)
    total_elims += solve_naked_pairs_group(candidates, sqr_group)

  return total_elims

In [10]:
def solve_naked_pairs_group(candidates, group):
  """
  Solves with the naked pairs strategy for a subsection of board

  candidates: 9x9x9 ndarray of booleans
  group: list of tuples (row, col)
  """

  pair_map = {}
  found_pairs = []
  elims = 0

  # Find naked pairs
  for r, c in group:
    cell_possibilites = []
    for d in range(9):
      if candidates[r, c, d]:
        cell_possibilites.append(d)
    if len(cell_possibilites) == 2:
      if tuple(cell_possibilites) in pair_map:
        found_pairs.append((pair_map[tuple(cell_possibilites)], (r, c)))
      else:
        pair_map[(tuple(cell_possibilites))] = (r, c)

  # Remove found pairs
  for (cell1, cell2) in found_pairs:
    r1, c1 = cell1
    r2, c2 = cell2

    # Get shared candidates
    digits = [d for d in range(9) if candidates[r1, c1, d]]

    for r, c in group:
      if (r, c) != (r1, c1) and (r, c) != (r2, c2):
        for d in digits:
          if candidates[r, c, d]:
            candidates[r, c, d] = False
            elims += 1

  return elims



### Hidden Pairs

In [11]:
def solve_hidden_pairs(board, candidates):
  """
  Solves with the hidden pairs strategy.

  board: 9x9 ndarray of ints
  candidates: 9x9x9 ndarray of booleans
  """
  total_elims = 0

  for unit_index in range(9):
    row_group = [(unit_index, col) for col in range(9)]
    col_group = [(row, unit_index) for row in range(9)]
    sqr_group = CellGroup(unit_index // 3 * 3, unit_index % 3 * 3).sqr


    total_elims += solve_hidden_pairs_group(candidates, row_group)
    total_elims += solve_hidden_pairs_group(candidates, col_group)
    total_elims += solve_hidden_pairs_group(candidates, sqr_group)

  return total_elims


In [12]:
def solve_hidden_pairs_group(candidates, group):
  """
  Solves with the naked pairs strategy for a subsection of board

  candidates: 9x9x9 ndarray of booleans
  group: list of tuples (row, col
  """
  hidden_pairs = find_hidden_pairs(candidates, group)
  elims = 0
  for cell1, cell2, val1, val2 in hidden_pairs:
    r1, c1 = cell1
    r2, c2 = cell2
    for i in range(9):
      if i != val1 and i != val2:
        if candidates[r1, c1, i]:
          candidates[r1, c1, i] = False
          elims += 1

        if candidates[r2, c2, i]:
          candidates[r2, c2, i] = False
          elims += 1

  return elims

In [13]:
def find_hidden_pairs(candidates, group):
  """
  Finds hidden pairs in a subsection of the board.

  candidates: 9x9x9 ndarray of booleans
  group: list of tuples (row, col)
  returns: list of tuples with element in form: (location(r, c), location(r, c), val1, val2)
  """
  # Get values with 2 possible locations
  possible_vals = [] # Holds (val, [(r,c), (r, c)]) of any value with 2 possible cells
  for val in range(9):
    possible_cells = get_possible_cells(candidates, group, val)
    if len(possible_cells) == 2:
      possible_vals.append((val, possible_cells))

  # Finds values that share same 2 possible locations
  matcher = {}
  for val, possible_cells in possible_vals:
    if tuple(possible_cells) in matcher:
      matcher[tuple(possible_cells)].append(val)
    else:
      matcher[tuple(possible_cells)] = [val]

  hidden_pairs = [] # Holds found hidden pairs
  for cell_pair, vals in matcher.items():
    if len(vals) == 2:
      hidden_pairs.append((cell_pair[0], cell_pair[1], vals[0], vals[1]))

  return hidden_pairs



In [14]:
def get_possible_cells(candidates, group, val):
  """
  Determines which cells in group a number can be placed.

  candidates: 9x9x9 ndarray of booleans
  group: list of tuples (row, col)
  val: int (0-8)
  """
  possible_cells = []

  for r, c in group:
    if candidates[r, c, val]:
      possible_cells.append((r, c))

  return possible_cells

# Solve Puzzle Tracking Methods Used

In [16]:
def solve_puzzle(puzzle):
  """
  Solves a sudoku puzzle.

  puzzle: 9x9 ndarray of ints
  """
  candidates = init_candidates(puzzle)
  methods_used = np.zeros(3, dtype=int)
  num_empty_cells = np.count_nonzero(puzzle == 0)

  iterations = 0

  while num_empty_cells > 0:
    iterations += 1

    if iterations > 200:
      return False

    found_naked = solve_naked_singles(puzzle, candidates)
    num_empty_cells -= found_naked
    if num_empty_cells == 0:
      break

    found_hidden = solve_hidden_singles(puzzle, candidates)


    if found_hidden > 0:
      num_empty_cells -= found_hidden
      methods_used[0] += 1
      continue
    if num_empty_cells == 0:
      break

    found_naked_pairs = solve_naked_pairs(puzzle, candidates)
    if found_naked_pairs > 0:
      methods_used[1] += 1
      continue

    found_hidden_pairs = solve_hidden_pairs(puzzle, candidates)
    if found_hidden_pairs > 0:
      methods_used[2] += 1
      continue



  return methods_used






# Test

## Find difference between two ndarrays

In [17]:
def visualize_diff(arr1, arr2):
  """
  Visualizes the difference between two 9x9x9 numpy bool arrays.
  Outputs the positions where the arrays differ.

  arr1, arr2: 9x9x9 numpy arrays of booleans
  """
  # Ensure both arrays are of the same shape
  if arr1.shape != arr2.shape:
    raise ValueError("Both arrays must have the same shape.")

  # Compare the two arrays and get the indices where they differ
  diff_indices = np.where(arr1 != arr2)

  # Get the row, column, and value indices of the differences
  diff_positions = list(zip(diff_indices[0], diff_indices[1], diff_indices[2]))

  if not diff_positions:
    print("No differences found.")
  else:
    print(f"Differences found at {len(diff_positions)} positions:")
    for position in diff_positions:
      print(f"Position: {position}, Value in arr1: {arr1[position]}, Value in arr2: {arr2[position]}")

### Test Hidden Pairs

Puzzle with hidden pair at (3, 4) and (4, 5)

In [17]:
hid_pair_puzzle = np.array([
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
  ])

In [ ]:
hid_pair_cands = init_candidates(hid_pair_puzzle)
orig_cands = hid_pair_cands.copy()

In [ ]:
elims = solve_hidden_pairs(hid_pair_puzzle, hid_pair_cands)
elims

In [ ]:
visualize_diff(orig_cands, hid_pair_cands)

### Solve with only naked/hidden singles

In [ ]:
puzzle_1 = np.array([
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 8, 0, 6, 0, 0, 3],
    [4, 0, 0, 8, 3, 0, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
])

## Get Puzzles from database

In [23]:
base_ngrok_url = 'https://99a1-2600-6c54-4e00-120d-fc5e-66fd-33ec-2f38.ngrok-free.app'
endpoint_url = '/api/puzzles'
url = f'{base_ngrok_url}{endpoint_url}'
response = requests.get(url)
data = []

if response.status_code != 200:
  print(f'API Call Error: {response.status_code}')
else:
  data = response.json()

print(data[0])
len(data)

{'puzzleId': 1, 'puzzleVals': '000260701680070090190004500820100040004602900050003028009300074040050036703018000', 'solutionVals': '435269781682571493197834562826195347374682915951743628519326874248957136763418259'}


102

# Attempt to solve puzzles with human strategies

## Attempt to solve all puzzles and record data

In [24]:
total_methods_used = []
failed_to_solve = 0
failed_puzzles = []
for puzzle_json in data:
  puzzle = np.array(list(puzzle_json['puzzleVals']), dtype=int).reshape((9, 9))
  result = solve_puzzle(puzzle)
  if result is False:
    failed_to_solve += 1
    print('couldnt solve')
    failed_puzzles.append(puzzle)
  else:
    print('Solved!')
    total_methods_used.append(result)
print(f'Failed to solve {failed_to_solve} puzzles\n')
print(total_methods_used)

Solved!
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solve
couldnt solv

In [21]:
print(failed_puzzles)

[array([[0, 0, 1, 2, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 7, 0, 1, 0, 0, 4],
       [0, 0, 0, 3, 7, 6, 0, 0, 2],
       [0, 0, 0, 8, 1, 9, 4, 5, 3],
       [0, 0, 0, 4, 2, 5, 6, 0, 7],
       [0, 0, 0, 0, 4, 3, 0, 7, 0],
       [5, 4, 3, 0, 0, 7, 0, 2, 0],
       [0, 0, 0, 0, 0, 2, 3, 4, 6]]), array([[0, 8, 0, 0, 0, 0, 0, 9, 0],
       [0, 0, 5, 0, 0, 0, 1, 2, 0],
       [0, 0, 9, 0, 0, 0, 8, 4, 0],
       [0, 0, 0, 0, 9, 0, 4, 5, 0],
       [0, 0, 4, 0, 0, 0, 0, 0, 2],
       [0, 0, 0, 0, 0, 4, 0, 0, 9],
       [0, 0, 1, 0, 0, 2, 0, 3, 0],
       [0, 0, 6, 3, 5, 0, 9, 0, 0],
       [0, 5, 3, 0, 6, 0, 2, 0, 0]]), array([[4, 5, 0, 2, 7, 0, 0, 0, 3],
       [0, 0, 0, 0, 4, 5, 0, 0, 8],
       [0, 0, 0, 1, 0, 0, 0, 4, 5],
       [0, 0, 0, 4, 0, 0, 0, 5, 0],
       [0, 4, 0, 9, 5, 7, 8, 0, 0],
       [0, 0, 5, 0, 0, 0, 4, 9, 0],
       [5, 0, 0, 0, 0, 0, 0, 0, 0],
       [8, 0, 0, 5, 0, 0, 0, 0, 0],
       [0, 0, 4, 7, 1, 3, 5, 8, 6]]), array([[1, 0, 0, 3, 9